# BATDiff on OCT: bicubic reference vs DIP reference

I run the same BATDiff model twice on one OCT B-scan and compare the results.

| Run | Reference image `x_ref` | Role |
|---|---|---|
| A — baseline | LR stretched up with bicubic (upstream behaviour) | published method |
| B — contribution | Deep Image Prior reconstruction | this project |

BATDiff decomposes `x_ref` with an a-trous wavelet transform and trains a diffusion model to reproduce the per-scale detail planes. Changing the reference therefore changes what the model learns, not only what happens to its output afterwards.

On this B-scan the finest wavelet plane carries about 74 times more detail energy under the DIP reference than under bicubic. With bicubic that plane is nearly empty, so the diffusion model has almost nothing to learn at the scale that matters most.

I set the runtime to T4 GPU. I need `outputs/oct/dip_full/` and `scripts/batdiff_dip_patch.py` in my GitHub repo, because this notebook pulls the images and the patch from there. Each full configuration takes about 1–3 hours depending on the settings below.


In [ ]:
#@title Step 0 — check the runtime has a GPU
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible.\n"
        "BATDiff hardcodes a CUDA device and will not run on CPU.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then run this cell again."
    )

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"VRAM:  {total_gb:.1f} GB")
print(f"torch: {torch.__version__}")

if total_gb < 14:
    print("\nWarning: with under ~14 GB you will need a smaller DIM in the config cell.")

## Step 1 — Fetch both repositories

BATDiff is the upstream method. My project repo supplies the OCT images from the DIP stage, the patch script, and the PSNR/SSIM/LPIPS code, so I score BATDiff with the same metric implementation as DIP.

I only install four extra packages on Colab. I do not install BATDiff's `requirements.txt`, because it pins `torch==1.13.1` and `numpy==1.24.1` and would break CUDA on this runtime.


In [ ]:
#@title Step 1 — clone repos and install the four missing packages
import os, subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/yoyowuyogwrt-hue/3D-OCT-Image-SuperResolution-Benchmark"
BATDIFF_REPO = "https://github.com/MaryamHeidari-1994/BATDiff"

WORK     = Path("/content")
BATDIFF  = WORK / "BATDiff"
PROJECT  = WORK / "project"


def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f"command failed with exit code {result.returncode}")


if not BATDIFF.exists():
    sh(f"git clone --depth 1 {BATDIFF_REPO} {BATDIFF}")
if not PROJECT.exists():
    sh(f"git clone --depth 1 {PROJECT_REPO} {PROJECT}")

sh("pip install -q einops ftfy regex PyWavelets lpips")

print("\nBATDiff:", BATDIFF)
print("project:", PROJECT)

## Step 2 — Apply the patch

`scripts/batdiff_dip_patch.py` makes two kinds of edit, kept separate on purpose:

- the contribution — `create_img_scales()` gains an `xref_path` argument and `main.py` gains a `--xref_image` flag;
- compatibility — `clip/clip.py` imports `packaging` from `pkg_resources`, which setuptools 81 removed.

Both runs use this one patched checkout. Omitting `--xref_image` reproduces upstream behaviour, so the baseline and the contribution differ only in the reference image.


In [ ]:
#@title Step 2 — patch BATDiff, then show the changed lines
sh(f"python {PROJECT}/scripts/batdiff_dip_patch.py --batdiff-root {BATDIFF}")

print("=" * 70)
print("The patched reference block in BATDiff/functions.py:")
print("=" * 70)

source = (BATDIFF / "BATDiff" / "functions.py").read_text().splitlines()
start = next(i for i, line in enumerate(source) if "if xref_path is not None:" in line)
for offset, line in enumerate(source[start - 1:start + 11]):
    print(f"{start + offset:4d} | {line}")

## Step 3 — Stage the OCT images

Three images come from the DIP stage, all of slice 256 of `TX12_D0_A1L`:

| File | Size (W x H) | Used for |
|---|---|---|
| `lr.png` | 64 x 128 | the input BATDiff super-resolves |
| `hr.png` | 512 x 1024 | ground truth, evaluation only |
| `dip.png` | 512 x 1024 | the DIP reference for run B |

Each run gets its own dataset folder, because `create_img_scales(create=True)` writes `scale_0/` … `scale_5/` next to the input image. Sharing one folder would mix the two references.


In [ ]:
#@title Step 3 — stage the images and sanity-check their sizes
import shutil
from PIL import Image

SR_FACTOR = 8
DIP_DIR   = PROJECT / "outputs" / "oct" / "dip_full"

LR_SRC  = DIP_DIR / "lr.png"
HR_SRC  = DIP_DIR / "hr.png"
DIP_SRC = DIP_DIR / "dip.png"

missing = [p for p in (LR_SRC, HR_SRC, DIP_SRC) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing from the cloned project repo:\n  "
        + "\n  ".join(str(p) for p in missing)
        + "\n\nRun scripts/run_dip_oct_full.py locally, then commit and push "
          "outputs/oct/dip_full/."
    )

lr_size, hr_size, dip_size = (Image.open(p).size for p in (LR_SRC, HR_SRC, DIP_SRC))
expected_hr = (lr_size[0] * SR_FACTOR, lr_size[1] * SR_FACTOR)
assert hr_size == expected_hr, f"HR is {hr_size}, expected {expected_hr} for x{SR_FACTOR}"
assert dip_size == hr_size, f"DIP output is {dip_size}, expected {hr_size}"

DATA_BASE = WORK / "data"
DATA_BICUBIC = DATA_BASE / "bicubic"
DATA_DIP     = DATA_BASE / "dip"

for folder in (DATA_BICUBIC, DATA_DIP):
    folder.mkdir(parents=True, exist_ok=True)
    shutil.copy(LR_SRC, folder / "lr.png")

XREF_DIP = DATA_BASE / "dip_reference.png"
shutil.copy(DIP_SRC, XREF_DIP)

print(f"LR  {lr_size[0]}x{lr_size[1]}   ->  SR target {expected_hr[0]}x{expected_hr[1]}")
print(f"HR  {hr_size[0]}x{hr_size[1]}   (evaluation only, never given to BATDiff)")
print(f"DIP {dip_size[0]}x{dip_size[1]}   (reference for run B)")

## Step 4 — Configuration

I leave `SMOKE_TEST = True` for the first pass. It runs both configurations end to end in a few minutes at a tiny model size. The images look like noise; that only checks that the clone, the patch, the data, the two runs and the scoring work.

When the smoke test finishes cleanly, I set `SMOKE_TEST = False` and re-run from this cell.

I start at `DIM = 64`. Six full-resolution scales at 512x1024 do not fit a T4 at the upstream default of 200.


In [ ]:
#@title Step 4 — shared configuration for both runs
SMOKE_TEST = True  #@param {type:"boolean"}

if SMOKE_TEST:
    DIM, TRAIN_STEPS, TIMESTEPS = 16, 60, 20
else:
    DIM, TRAIN_STEPS, TIMESTEPS = 64, 4000, 100  #@param

ATROUS_LEVEL = 6
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# Identical for both runs; only --xref_image differs.
COMMON_FLAGS = (
    f"--mode train "
    f"--image_name lr.png "
    f"--use_atrous --atrous_wavelet b3 "
    f"--atrous_level {ATROUS_LEVEL} "
    f"--sr_factor {SR_FACTOR} "
    f"--dim {DIM} "
    f"--train_num_steps {TRAIN_STEPS} "
    f"--timesteps {TIMESTEPS} "
    f"--save_and_sample_every {max(TRAIN_STEPS, 1)} "
)

print(f"{'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}")
print(f"  dim           {DIM}")
print(f"  train steps   {TRAIN_STEPS}")
print(f"  timesteps     {TIMESTEPS}")
print(f"  atrous levels {ATROUS_LEVEL}")
if SMOKE_TEST:
    print("\nOutputs will look like noise. You are testing plumbing, not quality.")

## Step 5 — Runner

BATDiff writes one image per scale into `{results}/{scope}/final_samples/`, named `{timestep_list}_s{scale}_{description}.png`. The finest scale (the highest `_s{N}_`) is the super-resolved output. `run_batdiff()` streams the training log, then returns that finest-scale path.


In [ ]:
#@title Step 5 — runner
import re, sys, time, subprocess
from pathlib import Path


def find_final_image(scope_dir: Path) -> Path:
    """Return the finest-scale sample BATDiff wrote, i.e. the SR output."""
    candidates = list((scope_dir / "final_samples").glob("*.png"))
    if not candidates:
        raise FileNotFoundError(f"No samples under {scope_dir / 'final_samples'}")

    def scale_of(path: Path) -> int:
        match = re.search(r"_s(\d+)_", path.name)
        return int(match.group(1)) if match else -1

    finest = max(scale_of(p) for p in candidates)
    at_finest = [p for p in candidates if scale_of(p) == finest]
    return max(at_finest, key=lambda p: p.stat().st_mtime)


def run_batdiff(tag: str, dataset_folder: Path, xref: Path | None) -> Path:
    scope_dir = RESULTS / tag / tag
    flags = COMMON_FLAGS + f"--scope {tag} "
    flags += f"--dataset_folder {dataset_folder}/ --results_folder {RESULTS / tag} "
    if xref is not None:
        flags += f"--xref_image {xref} "

    print("=" * 70)
    print(f"RUN {tag}   reference = {'DIP output' if xref else 'bicubic (upstream)'}")
    print("=" * 70)

    started = time.time()
    process = subprocess.Popen(
        f"python main.py {flags}", shell=True, cwd=BATDIFF, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        sys.stdout.write(line)
    process.wait()

    elapsed = (time.time() - started) / 60
    if process.returncode:
        raise RuntimeError(f"run {tag} failed with exit code {process.returncode}")

    output = find_final_image(scope_dir)
    print(f"\nrun {tag} finished in {elapsed:.1f} min -> {output.name}")
    return output


print("Runner ready.")

In [ ]:
#@title Run A — baseline, bicubic reference (upstream behaviour)
out_bicubic = run_batdiff("bicubic", DATA_BICUBIC, xref=None)

In [ ]:
#@title Run B — contribution, DIP reference
out_dip = run_batdiff("dip", DATA_DIP, xref=XREF_DIP)

## Step 6 — Score everything with one metric implementation

I score all four reconstructions against the same held-out HR B-scan using `src/metrics/evaluate.py` from my project repo — the same code that produced the DIP numbers.

The row that matters is BATDiff (DIP ref) against BATDiff (bicubic ref): same model, same hyperparameters, same input, one variable changed.

LPIPS often moves before PSNR. PSNR rewards blur; LPIPS is closer to whether the B-scan still looks like an eye scan.


In [ ]:
#@title Step 6 — evaluate
import sys, csv
import numpy as np
from PIL import Image

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics.evaluate import evaluate

hr = np.asarray(Image.open(HR_SRC).convert("RGB"))
target_size = (hr.shape[1], hr.shape[0])  # PIL order: (W, H)


def load_like_hr(path: Path) -> np.ndarray:
    image = Image.open(path).convert("RGB")
    if image.size != target_size:
        print(f"note: resizing {path.name} from {image.size} to {target_size} for scoring")
        image = image.resize(target_size, Image.BICUBIC)
    return np.asarray(image)


methods = {
    "Bicubic ×8":            DIP_DIR / "bicubic.png",
    "DIP alone":             DIP_SRC,
    "BATDiff (bicubic ref)": out_bicubic,
    "BATDiff (DIP ref)":     out_dip,
}

rows = []
for name, path in methods.items():
    scores = evaluate(hr, load_like_hr(path))
    rows.append({"method": name, **scores})

print(f"\n{'method':<24}{'PSNR ↑':>9}{'SSIM ↑':>9}{'LPIPS ↓':>10}")
print("-" * 52)
for row in rows:
    print(f"{row['method']:<24}{row['PSNR']:>9.4f}{row['SSIM']:>9.4f}{row['LPIPS']:>10.4f}")

base = next(r for r in rows if r["method"] == "BATDiff (bicubic ref)")
ours = next(r for r in rows if r["method"] == "BATDiff (DIP ref)")
print("\nContribution vs baseline (DIP ref − bicubic ref):")
print(f"  PSNR  {ours['PSNR']  - base['PSNR']:+.4f}   (positive is better)")
print(f"  SSIM  {ours['SSIM']  - base['SSIM']:+.4f}   (positive is better)")
print(f"  LPIPS {ours['LPIPS'] - base['LPIPS']:+.4f}   (negative is better)")

out_csv = RESULTS / ("scores_smoke.csv" if SMOKE_TEST else "scores.csv")
with out_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["method", "PSNR", "SSIM", "LPIPS"])
    writer.writeheader()
    writer.writerows(rows)
print(f"\nsaved {out_csv}")
if SMOKE_TEST:
    print("SMOKE TEST numbers are meaningless. Set SMOKE_TEST = False and re-run.")

In [ ]:
#@title Step 7 — comparison figure, and download the results
import matplotlib.pyplot as plt

panels = [("HR (ground truth)", hr)] + [
    (row["method"], load_like_hr(methods[row["method"]])) for row in rows
]

fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 8))
for axis, (title, image) in zip(axes, panels):
    axis.imshow(image)
    axis.set_title(title, fontsize=10)
    axis.axis("off")
    if title != "HR (ground truth)":
        row = next(r for r in rows if r["method"] == title)
        axis.set_xlabel("")
        axis.text(0.5, -0.03,
                  f"PSNR {row['PSNR']:.2f}\nSSIM {row['SSIM']:.3f}\nLPIPS {row['LPIPS']:.3f}",
                  transform=axis.transAxes, ha="center", va="top", fontsize=8)
fig.tight_layout()

figure_path = RESULTS / ("comparison_smoke.png" if SMOKE_TEST else "comparison.png")
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {figure_path}")

# Bundle everything so it can go back into the repo alongside the DIP outputs.
archive = shutil.make_archive(str(WORK / "batdiff_oct_results"), "zip", RESULTS)
print(f"archive: {archive}")
try:
    from google.colab import files
    files.download(archive)
except Exception as error:
    print(f"(automatic download unavailable: {error}; grab it from the Files pane)")

## If something fails

**CUDA out of memory** — I halve `DIM`, then Runtime → Restart runtime, and re-run from Step 1. Colab does not reliably release VRAM after an OOM.

**No GPU** — I set Runtime → Change runtime type → T4 GPU.

**Missing images from the cloned project repo** — `outputs/oct/dip_full/` has not been pushed. I commit and push it, delete `/content/project`, and run Step 1 again.

**Colab disconnects mid-run** — I keep the tab visible. Free Colab reclaims idle sessions.

I copy finished zips into `outputs/oct/batdiff/` on my Mac.
